In [ ]:
# 必要なライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from scipy.integrate import quad, solve_ivp
from scipy.optimize import minimize
import pandas as pd
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Any
import warnings
warnings.filterwarnings('ignore')

# プロット設定
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8')

# Google Colab環境での日本語フォント設定
try:
    import matplotlib
    matplotlib.font_manager.fontManager.addfont('/System/Library/Fonts/Hiragino Sans GB.ttc')
    plt.rcParams['font.family'] = 'Hiragino Sans GB'
except:
    # Colab環境でのフォント設定
    try:
        import subprocess
        subprocess.run(['apt-get', 'install', '-y', 'fonts-noto-cjk'], capture_output=True)
        plt.rcParams['font.family'] = 'Noto Sans CJK JP'
    except:
        print("日本語フォントの設定をスキップします")

print("✅ ライブラリの読み込み完了")


In [ ]:
# 基本物理定数と宇宙論パラメータ
class CosmologicalConstants:
    """宇宙論的定数の定義"""
    
    def __init__(self):
        # 基本物理定数（SI単位）
        self.c = 2.998e8        # m/s - 光速
        self.hbar = 1.055e-34   # J⋅s - 換算プランク定数 
        self.k_B = 1.381e-23    # J/K - ボルツマン定数
        self.G = 6.674e-11      # m³⋅kg⁻¹⋅s⁻² - 重力定数
        
        # 宇宙論パラメータ（Planck 2018）
        self.H_0 = 67.66        # km/s/Mpc - ハッブル定数
        self.h = self.H_0 / 100 # 無次元ハッブルパラメータ
        self.Omega_m = 0.3111   # 物質密度パラメータ
        self.Omega_b = 0.04897  # バリオン密度パラメータ
        self.Omega_cdm = self.Omega_m - self.Omega_b  # 暗黒物質密度
        self.Omega_Lambda = 0.6889  # 暗黒エネルギー密度
        self.n_s = 0.9665       # スカラースペクトル指数
        self.A_s = 2.105e-9     # 原始スペクトル振幅
        self.tau = 0.0561       # 再電離光学的厚み
        
        # 情報物理学パラメータ
        self.alpha_info = 0.075  # 情報補正係数
        self.beta_info = 0.02    # 相互作用係数
        self.gamma_info = 0.001  # 減衰係数
        
        # 計算用パラメータ
        self.R_8 = 8.0          # Mpc/h - σ₈測定半径
        self.k_max = 10.0       # h/Mpc - 最大波数
        self.k_min = 1e-4       # h/Mpc - 最小波数

# 定数クラスのインスタンス化
cosmo = CosmologicalConstants()

print("✅ 宇宙論パラメータ設定完了")
print(f"   H₀ = {cosmo.H_0:.2f} km/s/Mpc")
print(f"   Ω_m = {cosmo.Omega_m:.4f}")
print(f"   Ω_Λ = {cosmo.Omega_Lambda:.4f}")
print(f"   n_s = {cosmo.n_s:.4f}")
print(f"   情報補正 α = {cosmo.alpha_info:.3f}")


In [ ]:
class InformationPhysics:
    """情報物理学の核心機能"""
    
    def __init__(self, cosmo):
        self.cosmo = cosmo
        
    def cosmic_temperature_evolution(self, z):
        """宇宙温度の赤方偏移依存性"""
        T_CMB_0 = 2.725  # K - 現在のCMB温度
        return T_CMB_0 * (1 + z)
    
    def landauer_information_energy(self, n_bits, T):
        """ランダウアー原理によるエネルギー"""
        return n_bits * self.cosmo.k_B * T * np.log(2)
    
    def information_density_evolution(self, z):
        """情報密度の宇宙論的進化"""
        T = self.cosmic_temperature_evolution(z)
        # 情報エントロピーは宇宙の進化とともに増加
        S_info = 1e25 * (1 + z)**(-2)  # 経験的スケーリング
        
        # ハッブル体積当たりの情報密度
        H = self.cosmo.H_0 * np.sqrt(
            self.cosmo.Omega_m * (1 + z)**3 + self.cosmo.Omega_Lambda
        )
        V_hubble = (3e8 / (H * 1000))**3  # m³
        
        rho_I = self.landauer_information_energy(S_info, T) / V_hubble
        return rho_I
    
    def information_correction_factor(self, z):
        """情報密度による補正因子"""
        rho_I = self.information_density_evolution(z)
        
        # 臨界密度
        H = self.cosmo.H_0 * 1000 / 3.086e22  # s⁻¹
        rho_crit = 3 * H**2 / (8 * np.pi * self.cosmo.G)
        
        # 情報補正（相対的な寄与）
        delta_I = self.cosmo.alpha_info * (rho_I / rho_crit) * (1 + z)**(-1)
        
        return 1 + delta_I

# 情報物理学クラスのインスタンス化
info_physics = InformationPhysics(cosmo)

# テスト計算
z_test = np.array([0, 1, 10, 100, 1000])
T_test = [info_physics.cosmic_temperature_evolution(z) for z in z_test]
correction_test = [info_physics.information_correction_factor(z) for z in z_test]

print("✅ 情報物理学クラス初期化完了")
print("\n温度進化テスト:")
for z, T in zip(z_test, T_test):
    print(f"   z = {z:4.0f}: T = {T:8.2f} K")

print("\n情報補正因子テスト:")
for z, corr in zip(z_test, correction_test):
    print(f"   z = {z:4.0f}: correction = {corr:8.6f}")


In [ ]:
class Sigma8Calculator:
    """σ₈計算の核心クラス"""
    
    def __init__(self, cosmo, info_physics):
        self.cosmo = cosmo
        self.info_physics = info_physics
        
    def eisenstein_hu_transfer_with_information(self, k):
        """情報補正を含むEisenstein-Hu転移関数"""
        
        # 基本パラメータ
        theta_cmb = 2.7255 / 2.7  # CMB温度補正
        omega_m = self.cosmo.Omega_m * self.cosmo.h**2
        omega_b = self.cosmo.Omega_b * self.cosmo.h**2
        omega_cdm = omega_m - omega_b
        
        # 基本スケール
        k_eq = 7.46e-2 * omega_m / theta_cmb**2  # h/Mpc
        z_eq = 2.50e4 * omega_m / theta_cmb**4
        
        # バリオン・CDM比
        f_b = omega_b / omega_m
        f_c = omega_cdm / omega_m
        
        # 音響減衰
        alpha_c = (46.9 * omega_m / theta_cmb**2)**0.670 * (1 + (32.1 * omega_m / theta_cmb**2)**(-0.532))
        alpha_b = (46.9 * omega_m / theta_cmb**2)**0.670 * (1 + (458.0 * omega_m / theta_cmb**2)**(-0.708))
        beta_c = 1.0 / (1 + (0.944 / omega_m / theta_cmb**2)**0.395)
        beta_b = 0.5 + f_b + (3 - 2*f_b) * np.sqrt((17.2 * omega_m / theta_cmb**2)**2 + 1)
        
        # k依存性パラメータ
        q = k / (13.41 * k_eq)
        
        # CDM転移関数
        def T_0(k_val, alpha, beta):
            q_val = k_val / (13.41 * k_eq)
            L_0 = np.log(np.e + 1.8 * beta * q_val)
            C_0 = 14.2 / alpha + 386.0 / (1 + 69.9 * q_val**1.08)
            return L_0 / (L_0 + C_0 * q_val**2)
        
        T_c = T_0(k, alpha_c, beta_c)
        
        # バリオン転移関数
        T_b = T_0(k, alpha_b, beta_b)
        
        # 総転移関数
        T_total = f_b * T_b + f_c * T_c
        
        # 情報補正の適用
        z_recombination = 1090  # 再結合期
        info_correction = self.info_physics.information_correction_factor(z_recombination)
        
        # 情報物理学による修正
        k_info = k / info_correction  # 有効波数のシフト
        T_info_corrected = T_total * info_correction
        
        return T_info_corrected
    
    def linear_power_spectrum_with_information(self, k, z=0):
        """情報補正を含む線形パワースペクトル"""
        
        # 原始パワースペクトル
        P_primordial = self.cosmo.A_s * (k / 0.05)**(self.cosmo.n_s - 1)
        
        # 転移関数
        T_k = self.eisenstein_hu_transfer_with_information(k)
        
        # 成長因子（情報補正含む）
        D_z = self.growth_factor_with_information(z)
        
        # 線形パワースペクトル
        P_linear = P_primordial * T_k**2 * D_z**2
        
        return P_linear
    
    def growth_factor_with_information(self, z):
        """情報補正を含む成長因子"""
        
        # 線形成長因子の積分
        def growth_integrand(a_prime):
            z_prime = 1/a_prime - 1
            H_ratio_sq = (self.cosmo.Omega_m * a_prime**(-3) + 
                         self.cosmo.Omega_Lambda)
            # 情報補正
            info_factor = self.info_physics.information_correction_factor(z_prime)
            return info_factor / (a_prime * H_ratio_sq)**(3/2)
        
        # 積分実行
        a_z = 1 / (1 + z)
        integral, _ = quad(growth_integrand, 0, a_z)
        
        # 正規化（z=0での値で規格化）
        if z == 0:
            normalization = 1.0
        else:
            integral_0, _ = quad(growth_integrand, 0, 1.0)
            normalization = integral / integral_0
        
        return normalization
    
    def window_function_tophat(self, k, R):
        """トップハット窓関数"""
        x = k * R
        # x=0での特異点を回避
        result = np.where(x < 1e-3, 
                         1 - x**2/10 + x**4/280,  # テイラー展開
                         3 * (np.sin(x) - x * np.cos(x)) / x**3)
        return result
    
    def calculate_sigma_8_with_information(self):
        """情報補正を含むσ₈の計算"""
        
        print("🔄 σ₈計算を開始...")
        
        # 積分用波数グリッド
        k_values = np.logspace(np.log10(self.cosmo.k_min), 
                              np.log10(self.cosmo.k_max), 2000)
        
        # パワースペクトルの計算
        P_k = np.array([self.linear_power_spectrum_with_information(k) for k in k_values])
        
        # 窓関数の計算
        W_k = self.window_function_tophat(k_values, self.cosmo.R_8)
        
        # σ₈²の積分
        integrand = P_k * W_k**2 * k_values**2
        sigma_8_squared = np.trapz(integrand, k_values) / (2 * np.pi**2)
        
        sigma_8 = np.sqrt(sigma_8_squared)
        
        print(f"✅ σ₈計算完了: {sigma_8:.6f}")
        return sigma_8
    
    def compare_traditional_vs_information(self):
        """従来手法vs情報補正の比較"""
        
        print("🔄 比較計算を実行中...")
        
        # 従来手法（情報補正なし）
        original_alpha = self.cosmo.alpha_info
        self.cosmo.alpha_info = 0.0  # 情報補正を無効化
        
        sigma_8_traditional = self.calculate_sigma_8_with_information()
        
        # 情報補正あり
        self.cosmo.alpha_info = original_alpha  # 情報補正を復活
        sigma_8_information = self.calculate_sigma_8_with_information()
        
        # 観測値との比較
        sigma_8_observed = 0.8102  # Planck 2018
        
        error_traditional = abs(sigma_8_traditional - sigma_8_observed) / sigma_8_observed * 100
        error_information = abs(sigma_8_information - sigma_8_observed) / sigma_8_observed * 100
        
        improvement_factor = error_traditional / error_information
        
        results = {
            'traditional': sigma_8_traditional,
            'information': sigma_8_information,
            'observed': sigma_8_observed,
            'error_traditional': error_traditional,
            'error_information': error_information,
            'improvement_factor': improvement_factor
        }
        
        print(f"\n📊 比較結果:")
        print(f"   従来手法:     σ₈ = {sigma_8_traditional:.6f} (誤差: {error_traditional:.1f}%)")
        print(f"   情報補正:     σ₈ = {sigma_8_information:.6f} (誤差: {error_information:.1f}%)")
        print(f"   観測値:       σ₈ = {sigma_8_observed:.6f}")
        print(f"   改善率:       {improvement_factor:.1f}倍")
        
        return results

# σ₈計算クラスのインスタンス化
sigma8_calc = Sigma8Calculator(cosmo, info_physics)

print("✅ σ₈計算クラス初期化完了")


In [ ]:
# σ₈問題解決の実行
print("🚀 σ₈問題解決計算を開始")
print("=" * 50)

# 従来手法 vs 情報補正の比較計算
results = sigma8_calc.compare_traditional_vs_information()

print("\n" + "=" * 50)
print("📈 最終結果サマリー")
print("=" * 50)

print(f"🎯 目標達成状況:")
print(f"   精度改善: {results['error_traditional']:.1f}% → {results['error_information']:.1f}%")
print(f"   改善倍率: {results['improvement_factor']:.1f}倍")
print(f"   目標誤差(<5%): {'✅ 達成' if results['error_information'] < 5.0 else '❌ 未達成'}")

# 結果をDataFrameで整理
import pandas as pd

comparison_df = pd.DataFrame({
    '手法': ['従来手法', '情報補正', '観測値(Planck)'],
    'σ₈値': [results['traditional'], results['information'], results['observed']],
    '誤差(%)': [results['error_traditional'], results['error_information'], 0.0],
    '改善度': [1.0, results['improvement_factor'], np.nan]
})

print(f"\n📊 詳細比較表:")
print(comparison_df.to_string(index=False, float_format='%.6f'))


In [ ]:
# 結果の包括的可視化
def create_comprehensive_visualization(results):
    """σ₈問題解決結果の包括的可視化"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('σ₈問題解決 - 生成的情報物理学の成果', fontsize=16, fontweight='bold')
    
    # 1. σ₈値の比較バーチャート
    methods = ['従来手法', '情報補正', 'Planck観測']
    sigma8_values = [results['traditional'], results['information'], results['observed']]
    colors = ['lightcoral', 'lightblue', 'gold']
    
    bars = ax1.bar(methods, sigma8_values, color=colors, alpha=0.8, edgecolor='black')
    ax1.set_ylabel('σ₈値')
    ax1.set_title('σ₈値の比較')
    ax1.set_ylim(0.75, 0.95)
    ax1.grid(True, alpha=0.3)
    
    # 各バーに値を表示
    for bar, value in zip(bars, sigma8_values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{value:.6f}', ha='center', va='bottom', fontweight='bold')
    
    # 観測値ラインを表示
    ax1.axhline(y=results['observed'], color='red', linestyle='--', 
                label=f"Planck観測値 = {results['observed']:.4f}")
    ax1.legend()
    
    # 2. 誤差の改善を示すバーチャート
    methods_error = ['従来手法', '情報補正']
    errors = [results['error_traditional'], results['error_information']]
    colors_error = ['lightcoral', 'lightblue']
    
    bars_error = ax2.bar(methods_error, errors, color=colors_error, alpha=0.8, edgecolor='black')
    ax2.set_ylabel('誤差 (%)')
    ax2.set_title('精度改善の実証')
    ax2.grid(True, alpha=0.3)
    
    # 目標ライン（5%）を表示
    ax2.axhline(y=5.0, color='green', linestyle='--', label='目標誤差 5%')
    ax2.legend()
    
    # 改善倍率を表示
    for bar, error in zip(bars_error, errors):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{error:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # 改善倍率をテキストで表示
    ax2.text(0.5, max(errors) * 0.8, 
             f'改善倍率: {results["improvement_factor"]:.1f}倍', 
             ha='center', va='center', fontsize=14, fontweight='bold',
             bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
    
    # 3. 情報補正因子の赤方偏移依存性
    z_range = np.logspace(0, 3, 100)
    correction_factors = [info_physics.information_correction_factor(z) for z in z_range]
    
    ax3.semilogx(z_range, correction_factors, 'b-', linewidth=2, label='情報補正因子')
    ax3.set_xlabel('赤方偏移 z')
    ax3.set_ylabel('補正因子 (1 + δI)')
    ax3.set_title('情報補正の宇宙論的進化')
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    
    # 重要な時期をマーク
    ax3.axvline(x=1090, color='red', linestyle='--', alpha=0.7, label='再結合期')
    ax3.axvline(x=6, color='orange', linestyle='--', alpha=0.7, label='再電離期')
    ax3.legend()
    
    # 4. パワースペクトルの比較
    k_range = np.logspace(-3, 1, 100)
    
    # 従来のパワースペクトル
    cosmo.alpha_info = 0.0
    P_traditional = [sigma8_calc.linear_power_spectrum_with_information(k) for k in k_range]
    
    # 情報補正パワースペクトル
    cosmo.alpha_info = 0.075
    P_information = [sigma8_calc.linear_power_spectrum_with_information(k) for k in k_range]
    
    ax4.loglog(k_range, P_traditional, 'r--', linewidth=2, label='従来手法', alpha=0.7)
    ax4.loglog(k_range, P_information, 'b-', linewidth=2, label='情報補正')
    ax4.set_xlabel('波数 k [h/Mpc]')
    ax4.set_ylabel('パワースペクトル P(k)')
    ax4.set_title('パワースペクトルの比較')
    ax4.grid(True, alpha=0.3)
    ax4.legend()
    
    # σ₈測定スケールをマーク
    k_8 = 1.0 / cosmo.R_8
    ax4.axvline(x=k_8, color='green', linestyle=':', 
                label=f'σ₈スケール (k ≈ {k_8:.3f})')
    ax4.legend()
    
    plt.tight_layout()
    plt.show()
    
    return fig

# 可視化の実行
print("📊 結果の可視化を生成中...")
fig = create_comprehensive_visualization(results)

print("✅ 可視化完了")
